# Model Comparison & Ablation Study

This notebook systematically compares multiple model architectures for stroke classification.

**Objectives:**
1. Compare classical ML vs deep learning approaches
2. Evaluate different CNN backbones
3. Ablation study: Image vs Features vs Hybrid
4. Fine-tuning experiments
5. Analyze accuracy vs speed vs size trade-offs

**Models Evaluated:**
- Baseline: Random Forest, MLP
- CNN Backbones: MobileNetV3, EfficientNetB0, Custom CNN
- Ablation: Image-only, Features-only, Hybrid

## Setup

In [13]:
import sys
import time
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')

In [26]:
# Import our pipeline modules
from modifiers.utils.config import load_config
config = load_config("../config/config.yaml")
print(f"Classes: {config.classes}")

Classes: ['underline', 'box', 'curly', 'delete', 'boxshortcut', 'curlyshortcut', 'circleshortcut', 'none']


## Load and Prepare Data

In [20]:

def load_processed_data(processed_dir):
    # Try to load latest.npz first, fall back to most recent file
    latest_file = processed_dir / "processed_data.npz"
    
    if latest_file.exists():
        npz_file = latest_file
    else:
        # Find most recent .npz file
        npz_files = sorted(
            processed_dir.glob("processed_data_*.npz"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if not npz_files:
            raise FileNotFoundError(
                f"No processed data found in {processed_dir}. "
                "Run 'python scripts/dataset.py' first."
            )
        npz_file = npz_files[0]
    
    # Load arrays
    data = np.load(npz_file)
    images = data["images"]
    features = data["features"]
    labels = data["labels"]
    
    return images, features, labels

In [21]:
processed_dir =  Path.cwd().parent / "data" / "processed"

images, features, labels= load_processed_data(processed_dir)

In [29]:
from sklearn.model_selection import train_test_split

def split_data(images, labels, features, test_size, val_size, random_state):
    n_samples = len(labels)
    indices = np.arange(n_samples)
    
    # First split: train+val vs test (split indices only)
    idx_trainval, idx_test = train_test_split(
        indices,
        test_size=test_size,
        stratify=labels,
        random_state=random_state,
    )
    
    # Second split: train vs val (split indices only)
    idx_train, idx_val = train_test_split(
        idx_trainval,
        test_size=val_size,
        stratify=labels[idx_trainval],
        random_state=random_state,
    )
    
    # Now index into arrays (this creates views where possible)
    return {
        "X_train_img": images[idx_train],
        "X_train_feat": features[idx_train],
        "y_train": labels[idx_train],
        "X_val_img": images[idx_val],
        "X_val_feat": features[idx_val],
        "y_val": labels[idx_val],
        "X_test_img": images[idx_test],
        "X_test_feat": features[idx_test],
        "y_test": labels[idx_test],
    }

splits = split_data(
    images=images,
    labels=labels,
    features=features,
    test_size=config.data.test_size,
    val_size=config.data.val_size,
    random_state=config.data.random_state,
)

## Helper Functions

In [22]:
# Store results
results = []

def evaluate_model(model, X_test, y_test, model_name, is_keras=True):
    """Evaluate model and record metrics."""
    
    # Measure inference time
    start = time.time()
    if is_keras:
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    else:
        y_pred = model.predict(X_test)
    inference_time = (time.time() - start) / len(y_test) * 1000  # ms per sample
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    
    # Model size
    if is_keras:
        params = model.count_params()
        size_mb = params * 4 / (1024 * 1024)  # Approximate size in MB
    else:
        import joblib
        import tempfile
        with tempfile.NamedTemporaryFile() as f:
            joblib.dump(model, f.name)
            size_mb = Path(f.name).stat().st_size / (1024 * 1024)
        params = 0
    
    result = {
        'Model': model_name,
        'Accuracy': accuracy,
        'Params': params,
        'Size (MB)': round(size_mb, 2),
        'Inference (ms)': round(inference_time, 2),
    }
    results.append(result)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Params: {params:,}")
    print(f"  Size: {size_mb:.2f} MB")
    print(f"  Inference: {inference_time:.2f} ms/sample")
    
    return y_pred, accuracy


def get_keras_callbacks(patience=15):
    """Standard callbacks for Keras models."""
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=patience,
            restore_best_weights=True,
            verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0,
        ),
    ]


# 2. Evaluate Deep Learning - Image Only Models

## 2.1 Simple Custom CNN (Image Only)

In [27]:
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_simple_cnn(input_shape=(136, 136, 3), num_classes=10):
    """Build a simple custom CNN for baseline comparison."""
    inputs = Input(shape=input_shape)
    
    # Conv Block 1
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Conv Block 2
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Conv Block 3
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Dense layers
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='simple_cnn')
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building Simple CNN...")
simple_cnn = build_simple_cnn(num_classes=config.num_classes)
simple_cnn.summary()

Building Simple CNN...


Model: "simple_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 136, 136, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 136, 136, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 136, 136, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 68, 68, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 68, 68, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 68, 68, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 68, 68, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 34, 34, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     9,470,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,566,408 (36.49 MB)

 Trainable params: 9,565,960 (36.49 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
# Train Simple CNN
print("\nTraining Simple CNN...")

history_simple = simple_cnn.fit(
    splits['X_train_img'], splits['y_train'],
    validation_data=(splits['X_val_img'], splits['y_val']),
    epochs=50,
    batch_size=32,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
simple_pred, simple_acc = evaluate_model(
    simple_cnn,
    splits['X_test_img'],
    splits['y_test'],
    'Simple CNN (Images)',
)

## 2.2 MobileNetV3 (Image Only)

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import GlobalAveragePooling2D

def build_mobilenet_image_only(input_shape=(136, 136, 3), num_classes=10):
    """MobileNetV3 with image input only (no features)."""
    inputs = Input(shape=input_shape)
    
    backbone = MobileNetV3Small(
        include_top=False,
        input_tensor=inputs,
        weights='imagenet',
    )
    backbone.trainable = False
    
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='mobilenet_image_only')
    model.compile(
        optimizer=Adam(learning_rate=2e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building MobileNetV3 (Image Only)...")
mobilenet_img = build_mobilenet_image_only(num_classes=config.num_classes)
print(f"Parameters: {mobilenet_img.count_params():,}")

In [ ]:
# Train MobileNetV3 Image Only
print("Training MobileNetV3 (Image Only)...")

history_mobilenet_img = mobilenet_img.fit(
    splits['X_train_img'], splits['y_train'],
    validation_data=(splits['X_val_img'], splits['y_val']),
    epochs=50,
    batch_size=32,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
mobilenet_img_pred, mobilenet_img_acc = evaluate_model(
    mobilenet_img,
    splits['X_test_img'],
    splits['y_test'],
    'MobileNetV3 (Images)',
)

# 3. Hybrid Models (Image + Features)

## 3.1 MobileNetV3 Hybrid (Current Best)

In [ ]:
from src.models.hybrid import build_hybrid_model

print("Building MobileNetV3 Hybrid...")
mobilenet_hybrid = build_hybrid_model(
    input_shape=config.model.input_shape,
    num_classes=config.num_classes,
    feature_dim=features.shape[1],
    learning_rate=2e-4,
    backbone_trainable=False,
    use_se_attention=True,
)

In [ ]:
print("Training MobileNetV3 Hybrid...")

history_hybrid = mobilenet_hybrid.fit(
    {'img_input': splits['X_train_img'], 'feature_input': splits['X_train_feat']},
    splits['y_train'],
    validation_data=(
        {'img_input': splits['X_val_img'], 'feature_input': splits['X_val_feat']},
        splits['y_val'],
    ),
    epochs=50,
    batch_size=32,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
hybrid_pred, hybrid_acc = evaluate_model(
    mobilenet_hybrid,
    {'img_input': splits['X_test_img'], 'feature_input': splits['X_test_feat']},
    splits['y_test'],
    'MobileNetV3 Hybrid',
)

## 3.2 EfficientNetB0 Hybrid

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Concatenate, LayerNormalization

def build_efficientnet_hybrid(input_shape=(136, 136, 3), num_classes=10, feature_dim=10):
    """EfficientNetB0 hybrid model with feature fusion."""
    
    # Image branch
    img_input = Input(shape=input_shape, name='img_input')
    backbone = EfficientNetB0(
        include_top=False,
        input_tensor=img_input,
        weights='imagenet',
    )
    backbone.trainable = False
    
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Feature branch
    feat_input = Input(shape=(feature_dim,), name='feature_input')
    f = Dense(128, activation='relu')(feat_input)
    f = LayerNormalization()(f)
    f = Dropout(0.25)(f)
    f = Dense(64, activation='relu')(f)
    f = Dropout(0.2)(f)
    
    # Fusion
    combined = Concatenate()([x, f])
    combined = Dense(384, activation='relu')(combined)
    combined = BatchNormalization()(combined)
    combined = Dropout(0.35)(combined)
    combined = Dense(192, activation='relu')(combined)
    combined = Dropout(0.25)(combined)
    
    outputs = Dense(num_classes, activation='softmax')(combined)
    
    model = Model([img_input, feat_input], outputs, name='efficientnet_hybrid')
    model.compile(
        optimizer=Adam(learning_rate=2e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building EfficientNetB0 Hybrid...")
efficientnet_hybrid = build_efficientnet_hybrid(
    num_classes=config.num_classes,
    feature_dim=features.shape[1],
)
print(f"Parameters: {efficientnet_hybrid.count_params():,}")

In [ ]:
print("Training EfficientNetB0 Hybrid...")

history_effnet = efficientnet_hybrid.fit(
    {'img_input': splits['X_train_img'], 'feature_input': splits['X_train_feat']},
    splits['y_train'],
    validation_data=(
        {'img_input': splits['X_val_img'], 'feature_input': splits['X_val_feat']},
        splits['y_val'],
    ),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
effnet_pred, effnet_acc = evaluate_model(
    efficientnet_hybrid,
    {'img_input': splits['X_test_img'], 'feature_input': splits['X_test_feat']},
    splits['y_test'],
    'EfficientNetB0 Hybrid',
)